# 15 · Checkpoint: a filtered recipe finder
Level L2 · Part 2 · builds on lesson 14 (and lessons 07-13)

Needs: nothing. Everything runs in local mode (`QdrantClient(":memory:")`) on the saved vectors
in `data/`. One `needs-download` cell re-embeds the lesson's queries with MiniLM (about 90 MB,
cached once); when it is skipped the saved query cache is used.

## Goal
By the end of this lesson you can build `find("spicy but not too hot", vegetarian=True,
max_minutes=30)`: a recipe finder over 2,000 recipes that ranks by meaning, obeys hard filters
every time, rejects bad input with a clear error, and is checked by printed tests.

## The idea
A user's request mixes two kinds of wish. **Soft preferences** ("spicy", "comforting") are about
meaning, so the embedding ranks them. **Hard constraints** ("vegetarian", "under 30 minutes") are
facts that must hold for every result, so they go into a payload filter, never into the query
text. A finder is the small piece of code that splits the request that way: keyword arguments
become a `models.Filter`, the text becomes a query vector, and Qdrant does both in one call.
Think of a waiter: "something spicy" is a taste they interpret; "I'm vegetarian" is a rule they
must not break.

## Picture

```mermaid
flowchart LR
    U["find('spicy but not too hot',<br/>vegetarian=True, max_minutes=30)"] --> V["check the arguments<br/>clear errors"]
    V --> T["text -> embed_query<br/>384-number vector"]
    V --> F["keyword args -> build_filter<br/>models.Filter(must, must_not)"]
    T --> Q["query_points('recipes',<br/>query, query_filter, limit)"]
    F --> Q
    D["2,000 recipes<br/>uuid5(slug) ids, payload indexes"] --> Q
    Q --> R["page of recipes<br/>title, cuisine, minutes, score"]
```

## Requirements
Your finder is done when every box is ticked, with printed output to prove it:

- [ ] **R1 Load.** Load the 2,000 recipes and saved vectors with `uuid5(slug)` ids and payload indexes; check the count.
- [ ] **R2 Query vectors.** `embed_query(text)` uses a saved cache, or the model when it is loaded, and otherwise raises a clear `KeyError`.
- [ ] **R3 Filters.** `build_filter(vegetarian, max_minutes, cuisine, avoid)` returns a `models.Filter`, or `None` with no rules. `None` means "don't care"; `vegetarian=False` means "meat or fish only".
- [ ] **R4 Find.** `find(text, k=5, **filters)` returns up to `k` recipes, best first, each with title, cuisine, vegetarian, minutes and score.

Then make it safe and prove it works:

- [ ] **R5 Validate.** An unknown cuisine, a non-positive `max_minutes` or a bad `k` raises `ValueError` that says what is allowed.
- [ ] **R6 Test.** Print (a) that every hit obeys its filters over many queries and rule combinations, and (b) pass/fail for the top hit of a few labelled queries.
- [ ] **R7 "Spicy but not too hot".** Show what the model does with "not too hot", and handle it with a rule.

## Build it
Start from lesson 14: `recipes_2000.csv`, `recipe_embeddings_minilm_2000.npz`, `recipe_id` and
`make_collection`. Lesson 11 has every filter piece you need (`FieldCondition`, `MatchValue`,
`Range`, `MatchTextAny`, `must_not`). A useful order: R1, R3, R4 with a cached query, then R5,
R2, R6, R7. Write your finder in the cell below; everything after it is the reference solution.

In [1]:
# your code here
# Suggested signatures:
#   embed_query(text) -> list[float] of length 384
#   build_filter(vegetarian=None, max_minutes=None, cuisine=None, avoid=None) -> models.Filter | None
#   find(text, k=5, vegetarian=None, max_minutes=None, cuisine=None, avoid=None) -> list[dict]

## Reference solution
**Try the build yourself first.** The cells below are one working answer, requirement by
requirement. Compare your choices with them, not just your output.

### R1 · Load the recipes with stable ids and indexes
Read the file and its vectors, and check they line up row by row (lesson 14). Local mode prints a
`UserWarning` that payload indexes have no effect there (lesson 11); we silence that one message,
because the same code is meant to run against a server later.

In [2]:
import os, uuid, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")
warnings.filterwarnings("ignore", message="Payload indexes have no effect")
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_2000.csv")
saved = np.load("../data/recipe_embeddings_minilm_2000.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
assert (saved["slugs"] == recipes.slug.to_numpy()).all(), "vectors and CSV rows are out of step"
assert np.allclose(np.linalg.norm(doc_emb, axis=1), 1.0, atol=1e-5), "vectors are not unit length"
print("recipes:", recipes.shape, "| vectors:", doc_emb.shape, "| model:", MODEL)

recipes: (2000, 6) | vectors: (2000, 384) | model: all-MiniLM-L6-v2


Lesson 14's pieces, re-declared: `recipe_id` (uuid5 of the slug in our fixed namespace), the
payload, and `make_collection`. One addition: a full-text index on `description` (lesson 11),
because R7 filters on words in it.

In [3]:
RECIPE_NS = uuid.uuid5(uuid.NAMESPACE_URL, "https://example.com/recipes/")
INDEXES = {"cuisine": models.PayloadSchemaType.KEYWORD,
           "vegetarian": models.PayloadSchemaType.BOOL,
           "minutes": models.PayloadSchemaType.INTEGER,
           "description": models.TextIndexParams(type=models.TextIndexType.TEXT,
                                                 tokenizer=models.TokenizerType.WORD, lowercase=True)}

def recipe_id(slug):
    return str(uuid.uuid5(RECIPE_NS, slug))

def make_collection(client, name="recipes"):
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(name, vectors_config=models.VectorParams(
        size=doc_emb.shape[1], distance=models.Distance.COSINE))
    for field, kind in INDEXES.items():
        client.create_payload_index(name, field_name=field, field_schema=kind)
    return name

print("beef-stew ->", recipe_id("beef-stew"))

beef-stew -> 89acc474-1f23-5810-8b94-54de9b7a876e


Build the points and load them with `upload_points` (lesson 14), then check the count against the
file. Swapping `":memory:"` for `url="http://localhost:6333"` would run everything below against
a server (lesson 13).

In [4]:
def to_points(df, emb):
    return [models.PointStruct(
        id=recipe_id(r.slug), vector=emb[i].tolist(),
        payload={"slug": r.slug, "title": r.title, "description": r.description, "cuisine": r.cuisine,
                 "vegetarian": bool(r.vegetarian), "minutes": int(r.minutes)})
        for i, r in enumerate(df.itertuples())]

client = QdrantClient(":memory:")
make_collection(client)
client.upload_points("recipes", to_points(recipes, doc_emb), batch_size=256)
assert client.count("recipes").count == len(recipes), "count does not match the file"
print("points in 'recipes':", client.count("recipes").count)

points in 'recipes': 2000


### R2 · Query vectors, with or without the model
The finder's queries must go through the same model as the recipes (lesson 05). The bank below
is every query this lesson asks. The download cell embeds them and saves the cache; the course
ships the file it writes, so skipping it changes nothing.

In [5]:
model = None
FINDER_QUERIES = [
    "spicy but not too hot", "a hot curry", "something with mushrooms", "smoky barbecue flavour",
    "a chocolate dessert", "a warming soup", "fish for dinner", "creamy comfort food",
    "vegetarian curry under 30 minutes",
]
CACHE_FILE = "../data/query_embeddings_minilm_finder.npz"
print(len(FINDER_QUERIES), "queries in the bank")

9 queries in the bank


The model cell (skipped without `TUT_ENABLE=download`):

In [6]:
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL, device="cpu")
np.savez(CACHE_FILE, model=MODEL, queries=FINDER_QUERIES, query_emb=model.encode_query(FINDER_QUERIES))
print("saved", len(FINDER_QUERIES), "query embeddings from", MODEL)

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


`embed_query` is lesson 06's: cache first, then the model if loaded, otherwise a `KeyError` that
says what to do. It refuses a cache made by another model.

In [7]:
cache = np.load(CACHE_FILE)
assert str(cache["model"]) == MODEL, f"query cache from {cache['model']}, recipes from {MODEL}"
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"], strict=True))

def embed_query(text):
    if text in query_cache:
        return query_cache[text].tolist()
    if model is not None:
        return model.encode_query(text).tolist()
    raise KeyError(f"{text!r} is not in the saved query cache and the model is not loaded")

print("cached:", len(query_cache), "| vector length:", len(embed_query("a hot curry")))

cached: 9 | vector length: 384


A question that is not in the cache, with the model not loaded, fails with that message:

In [8]:
try:
    embed_query("a new question")
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

KeyError: "'a new question' is not in the saved query cache and the model is not loaded"


### R3 · Turn keyword arguments into a filter
One `FieldCondition` per rule the caller gave (lesson 11). The test is `is not None`, so
`vegetarian=False` still becomes a rule (Common mistakes shows what `if vegetarian:` does).
`avoid` lists words that must not appear in the description: a `must_not` with `MatchTextAny`.
With no rules at all, the function returns `None`, which `query_points` reads as "no filter".

In [9]:
def build_filter(vegetarian=None, max_minutes=None, cuisine=None, avoid=None):
    must, must_not = [], []
    if vegetarian is not None:
        must.append(models.FieldCondition(key="vegetarian", match=models.MatchValue(value=vegetarian)))
    if max_minutes is not None:
        must.append(models.FieldCondition(key="minutes", range=models.Range(lte=max_minutes)))
    if cuisine is not None:
        must.append(models.FieldCondition(key="cuisine", match=models.MatchValue(value=cuisine)))
    if avoid:
        must_not.append(models.FieldCondition(key="description",
                                              match=models.MatchTextAny(text_any=" ".join(avoid))))
    if not must and not must_not:
        return None
    return models.Filter(must=must or None, must_not=must_not or None)

print(build_filter())
print(build_filter(vegetarian=True, max_minutes=30).model_dump(exclude_none=True))

None
{'must': [{'key': 'vegetarian', 'match': {'value': True}}, {'key': 'minutes', 'range': {'lte': 30.0}}]}


No arguments gives `None`; two arguments give a `Filter` with two `must` conditions (printed
with `model_dump(exclude_none=True)` to hide the empty fields).
`count(count_filter=...)` tells how many recipes pass a filter before any ranking, which is a
quick way to see how much a rule narrows the menu:

In [10]:
for rules in [{}, {"vegetarian": True}, {"vegetarian": True, "max_minutes": 30},
              {"vegetarian": False, "cuisine": "Thai"}]:
    n = client.count("recipes", count_filter=build_filter(**rules)).count
    print(f"{n:>5} recipes pass {rules}")

 2000 recipes pass {}
 1106 recipes pass {'vegetarian': True}
  544 recipes pass {'vegetarian': True, 'max_minutes': 30}


   51 recipes pass {'vegetarian': False, 'cuisine': 'Thai'}


### R4 · `find`
Text to vector, rules to filter, one `query_points` call, then a plain dict per hit so callers
never touch Qdrant objects. `show` prints a page.

In [11]:
FIELDS = ["title", "cuisine", "vegetarian", "minutes"]

def find(text, k=5, vegetarian=None, max_minutes=None, cuisine=None, avoid=None):
    flt = build_filter(vegetarian, max_minutes, cuisine, avoid)
    hits = client.query_points("recipes", query=embed_query(text), query_filter=flt,
                               limit=k, with_payload=FIELDS).points
    return [{**h.payload, "score": round(h.score, 3)} for h in hits]

def show(page, label):
    print(label)
    for r in page:
        veg = "veg" if r["vegetarian"] else "   "
        print(f"  {r['score']:.3f}  {r['title']:<34}{r['cuisine']:<15}{veg} {r['minutes']:>3} min")

show(find("a hot curry"), "find('a hot curry'):")
show(find("a hot curry", vegetarian=True, max_minutes=30), "...vegetarian=True, max_minutes=30:")

find('a hot curry'):
  0.592  Vindaloo                          Indian              70 min
  0.560  Spicy tofu green curry            Thai           veg  25 min
  0.553  Spicy pork green curry            Thai                45 min
  0.548  Spicy tempeh green curry          Thai           veg  45 min
  0.546  Spicy beef green curry            Thai                30 min


...vegetarian=True, max_minutes=30:
  0.560  Spicy tofu green curry            Thai           veg  25 min
  0.536  Smoky tofu green curry            Thai           veg  30 min
  0.503  Herby tempeh green curry          Thai           veg  30 min
  0.502  Creamy tempeh curry               Indian         veg  30 min
  0.482  Spicy spinach green curry         Thai           veg  25 min


Without rules the top curry is Vindaloo (pork, 70 minutes). With the two rules every hit is
vegetarian and at most 30 minutes, and the page is still full: Qdrant ranks only the 544
recipes that pass (pre-filtering, lesson 11). Three rules at once work the same way:

In [12]:
show(find("smoky barbecue flavour", vegetarian=False, cuisine="Mexican", max_minutes=40),
     "find('smoky barbecue flavour', vegetarian=False, cuisine='Mexican', max_minutes=40):")

find('smoky barbecue flavour', vegetarian=False, cuisine='Mexican', max_minutes=40):
  0.624  Smoky beef tacos                  Mexican             20 min
  0.604  Smoky pork tacos                  Mexican             35 min
  0.601  Smoky lamb tacos                  Mexican             15 min
  0.594  Smoky chicken tacos               Mexican             30 min
  0.589  Smoky turkey tacos                Mexican             35 min


### R5 · Reject bad input with a clear message
A keyword filter is an exact, case-sensitive match: `cuisine="thai"` matches nothing and the user
just sees an empty page (Common mistakes, 3). Check the arguments first, against the values that
exist in the data, and say what is allowed.

In [13]:
CUISINES = sorted(recipes.cuisine.unique())

def check_args(k, max_minutes, cuisine):
    if not (isinstance(k, int) and 1 <= k <= 50):
        raise ValueError(f"k must be a whole number from 1 to 50, got {k!r}")
    if max_minutes is not None and not (isinstance(max_minutes, int) and max_minutes > 0):
        raise ValueError(f"max_minutes must be a positive whole number, got {max_minutes!r}")
    if cuisine is not None and cuisine not in CUISINES:
        raise ValueError(f"unknown cuisine {cuisine!r}; choose from {CUISINES}")

def find(text, k=5, vegetarian=None, max_minutes=None, cuisine=None, avoid=None):
    check_args(k, max_minutes, cuisine)
    flt = build_filter(vegetarian, max_minutes, cuisine, avoid)
    hits = client.query_points("recipes", query=embed_query(text), query_filter=flt,
                               limit=k, with_payload=FIELDS).points
    return [{**h.payload, "score": round(h.score, 3)} for h in hits]

print(len(CUISINES), "cuisines:", CUISINES)

14 cuisines: ['American', 'British', 'Chinese', 'French', 'Greek', 'Indian', 'Italian', 'Japanese', 'Mediterranean', 'Mexican', 'Middle Eastern', 'Nordic', 'Spanish', 'Thai']


`find` is the same as before with one line added at the top. A misspelt cuisine now fails loudly
instead of returning nothing:

In [14]:
try:
    find("a hot curry", cuisine="thai")
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: unknown cuisine 'thai'; choose from ['American', 'British', 'Chinese', 'French', 'Greek', 'Indian', 'Italian', 'Japanese', 'Mediterranean', 'Mexican', 'Middle Eastern', 'Nordic', 'Spanish', 'Thai']


It prints `ValueError: unknown cuisine 'thai'; choose from [...]` with the 14 real names, so the
caller (or a dropdown in a web page) knows what to send. A time limit of zero fails too:

In [15]:
try:
    find("a hot curry", max_minutes=0)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: max_minutes must be a positive whole number, got 0


It prints `ValueError: max_minutes must be a positive whole number, got 0`; `k=0` is rejected the
same way.

### R6 · Test it
**(a) Filters are never broken.** For every query in the bank and a grid of rule combinations,
check every hit against its rules, and check the page is full whenever enough recipes pass.

In [16]:
GRID = [{}, {"vegetarian": True}, {"vegetarian": False}, {"max_minutes": 20},
        {"vegetarian": True, "max_minutes": 30}, {"cuisine": "Indian"},
        {"cuisine": "Italian", "vegetarian": False, "max_minutes": 30}]

def obeys(r, vegetarian=None, max_minutes=None, cuisine=None, avoid=None):
    return ((vegetarian is None or r["vegetarian"] == vegetarian)
            and (max_minutes is None or r["minutes"] <= max_minutes)
            and (cuisine is None or r["cuisine"] == cuisine))

checked = broken = short = 0
for text in FINDER_QUERIES:
    for rules in GRID:
        page = find(text, k=10, **rules)
        checked += len(page)
        broken += sum(not obeys(r, **rules) for r in page)
        short += len(page) < min(10, client.count("recipes", count_filter=build_filter(**rules)).count)
print(f"{len(FINDER_QUERIES) * len(GRID)} searches, {checked} hits checked: "
      f"{broken} broke a rule, {short} pages came back short")

63 searches, 630 hits checked: 0 broke a rule, 0 pages came back short


Zero broken and zero short: the rules hold for all 630 hits. This test says nothing about
whether the hits are *good*, only that they are *allowed*. That is the job of (b).

**(b) The top hit is right.** For a few queries, write down what a good first answer looks like,
as a word that must appear in its title.

In [17]:
labelled = [
    ("something with mushrooms", {}, "mushroom"),
    ("a chocolate dessert", {}, "chocolate"),
    ("smoky barbecue flavour", {"vegetarian": True}, "smoky"),
    ("a hot curry", {"vegetarian": True, "max_minutes": 30}, "curry"),
    ("fish for dinner", {"max_minutes": 30}, "tuna"),
    ("a warming soup", {"vegetarian": True, "max_minutes": 30}, "soup"),
]
passed = 0
for text, rules, word in labelled:
    top = find(text, k=1, **rules)[0]
    ok = word in top["title"].lower()
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {text:<26}{str(rules):<42}top: {top['title']}")
print(f"\n{passed} of {len(labelled)} passed")

PASS  something with mushrooms  {}                                        top: Smoky mushroom skewers
PASS  a chocolate dessert       {}                                        top: Chocolate brownies
PASS  smoky barbecue flavour    {'vegetarian': True}                      top: Smoky black bean green curry
PASS  a hot curry               {'vegetarian': True, 'max_minutes': 30}   top: Spicy tofu green curry
PASS  fish for dinner           {'max_minutes': 30}                       top: Lemon garlic tuna stir-fry


FAIL  a warming soup            {'vegetarian': True, 'max_minutes': 30}   top: Gazpacho

5 of 6 passed


Five of six pass, but treat the passes with care. The "smoky" label is weak: the query word is
in many titles, so "Smoky black bean green curry" passes without being barbecue at all.
Hand-picked title words prove little; lesson 16 measures quality properly. The failure is worth
reading: the top "warming soup" among quick vegetarian
recipes is Gazpacho, a *chilled* soup. Count how many vegetarian soups take 30 minutes or less:

In [18]:
soup_rules = build_filter(vegetarian=True, max_minutes=30)
soup_rules.must.append(models.FieldCondition(key="description", match=models.MatchText(text="soup")))
quick_soups = client.scroll("recipes", scroll_filter=soup_rules, limit=20, with_payload=FIELDS)[0]
print(len(quick_soups), "quick vegetarian recipes mention 'soup':", [p.payload["title"] for p in quick_soups])
show(find("a warming soup", k=1, vegetarian=True, max_minutes=30), "with the time limit:")
show(find("a warming soup", k=3, vegetarian=True), "without the time limit:")

1 quick vegetarian recipes mention 'soup': ['Gazpacho']
with the time limit:
  0.567  Gazpacho                          Spanish        veg  15 min
without the time limit:
  0.707  Herby cauliflower soup            French         veg  50 min
  0.702  Herby tempeh soup                 French         veg  45 min
  0.678  Creamy cauliflower soup           French         veg  60 min


Only Gazpacho passes all three rules, so the finder returned the best *allowed* recipe, even
though it is the opposite of warming. Without the time limit, warm soups come back (45-60
minutes). A filter can leave no good answer, and `find` still answers: the scores (Gazpacho
0.567 vs 0.707 for the best warm soup) are the only hint. Lesson 10's `score_threshold` and proper evaluation (lesson 16) deal
with this; for now, the test caught it.

### R7 · "Spicy but not too hot"
First, the call from the syllabus, as it is:

In [19]:
show(find("spicy but not too hot", vegetarian=True, max_minutes=30),
     "find('spicy but not too hot', vegetarian=True, max_minutes=30):")

find('spicy but not too hot', vegetarian=True, max_minutes=30):
  0.587  Spicy tofu stir-fry               Chinese        veg  20 min
  0.571  Spicy spinach stir-fry            Chinese        veg  30 min
  0.570  Spicy tofu salad                  Mediterranean  veg  10 min
  0.570  Spicy cauliflower stir-fry        Chinese        veg  15 min
  0.569  Spicy tempeh burger               American       veg  20 min


Every hit is quick and vegetarian, so the rules held. But all five are "Spicy ..." recipes, and
four of them say "fiery chili": exactly what "not too hot" asked to avoid. Embedding models
handle negation poorly: "spicy" and "hot" pull the vector towards spicy food, and "not" barely
moves it. Compare the query's average similarity to the "Spicy ..." and the "Smoky ..." recipes:

In [20]:
a = np.array(embed_query("spicy but not too hot"))
b = doc_emb[recipes.title.str.startswith("Spicy").to_numpy()]
c = doc_emb[recipes.title.str.startswith("Smoky").to_numpy()]
print(f"mean similarity to the {len(b)} 'Spicy ...' recipes: {(b @ a).mean():.3f}")
print(f"mean similarity to the {len(c)} 'Smoky ...' recipes: {(c @ a).mean():.3f}")

mean similarity to the 312 'Spicy ...' recipes: 0.500
mean similarity to the 326 'Smoky ...' recipes: 0.367


The query sits closer to the "fiery" recipes than to the milder smoky ones. So "not too hot" is
a rule, not a taste, and it belongs in the filter. In this data the hot recipes are the ones whose
description says "fiery", so `avoid=["fiery"]` removes them and keeps "spicy" in the text:

In [21]:
print(client.count("recipes", count_filter=build_filter(vegetarian=True, max_minutes=30)).count,
      "->", client.count("recipes", count_filter=build_filter(True, 30, avoid=["fiery"])).count,
      "recipes after avoid=['fiery']")
show(find("spicy but not too hot", vegetarian=True, max_minutes=30, avoid=["fiery"]),
     "...avoid=['fiery']:")

544 -> 466 recipes after avoid=['fiery']
...avoid=['fiery']:
  0.587  Spicy tofu stir-fry               Chinese        veg  20 min
  0.481  Smoky tofu stir-fry               Chinese        veg  25 min
  0.478  Smoky spinach stir-fry            Chinese        veg  15 min
  0.471  Smoky spinach fried rice          Chinese        veg  10 min
  0.469  Smoky tempeh fried rice           Chinese        veg  15 min


Now the page is the one fiery-free spicy dish (Spicy tofu stir-fry: "a hot chili garlic sauce",
from the original 50) followed by smoky paprika dishes: spicy, but not hot. The word list is
blunt, though. Adding "hot" removes more than you want, because "hot" also means "a hot wok":

In [22]:
for avoid in [["fiery"], ["fiery", "hot"]]:
    page = find("spicy but not too hot", k=3, vegetarian=True, max_minutes=30, avoid=avoid)
    print(f"avoid={avoid}:", [r["title"] for r in page])

avoid=['fiery']: ['Spicy tofu stir-fry', 'Smoky tofu stir-fry', 'Smoky spinach stir-fry']


avoid=['fiery', 'hot']: ['Smoky spinach fried rice', 'Smoky tempeh fried rice', 'Mango salsa']


With "hot" added, Spicy tofu stir-fry goes (fair: "hot chili"), but so do Smoky tofu and Smoky
spinach stir-fry, because they are "tossed in a hot wok". A
real catalogue would store heat as its own field (`heat: 1-5`) and filter on a range; words in a
description are a stand-in. The lesson holds either way: meaning goes in the query, rules go in
the filter.

## What just happened
- R1 loaded 2,000 points with `uuid5(slug)` ids and four payload indexes, and the count matched the file.
- R3-R4: keyword arguments became `must` / `must_not` conditions; `count` showed 2,000 → 544 for vegetarian and ≤ 30 min, and `find` ranked only those.
- R5: `cuisine="thai"` raised a `ValueError` listing the 14 real cuisines instead of silently returning nothing.
- R6: 630 hits over 63 searches broke no rule and none came back short; 5 of 6 labelled top hits passed, and the one failure (Gazpacho) was the only allowed soup.
- R7: the model ignored "not too hot" (4 of 5 hits were fiery); `avoid=["fiery"]` turned that phrase into a rule, and adding "hot" over-filtered.

## Common mistakes

**1. Testing a flag with `if vegetarian:`.** `False` is falsy, so asking for meat dishes adds no
rule at all and vegetarian recipes come back:

In [23]:
try:
    def build_filter_buggy(vegetarian=None):
        must = []
        if vegetarian:                                              # BUG: False is skipped
            must.append(models.FieldCondition(key="vegetarian", match=models.MatchValue(value=vegetarian)))
        return models.Filter(must=must) if must else None

    hits = client.query_points("recipes", query=embed_query("something with mushrooms"),
                               query_filter=build_filter_buggy(vegetarian=False), limit=5).points
    print([h.payload["title"] for h in hits])
    assert not any(h.payload["vegetarian"] for h in hits), "asked for no vegetarian recipes, got some"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

['Smoky mushroom skewers', 'Herby mushroom skewers', 'Herby mushroom burger', 'Spicy mushroom pie', 'Herby mushroom green curry']
AssertionError: asked for no vegetarian recipes, got some


Every mushroom dish is vegetarian, so it prints `AssertionError: asked for no vegetarian recipes,
got some`. Fix: `if vegetarian is not None:`, as `build_filter` does. With it, the same request
returns meat or fish dishes only:

In [24]:
print([r["title"] for r in find("something with mushrooms", vegetarian=False)])

['Smoky duck pasta', 'Smoky cod pasta', 'Spaghetti carbonara', 'Smoky duck stew', 'Herby duck pasta']


**2. Writing the rules into the query text.** It feels natural to search for "vegetarian curry
under 30 minutes". The model sees a sentence about curry, not two rules:

In [25]:
try:
    page = find("vegetarian curry under 30 minutes")
    show(page, "rules in the text:")
    assert all(r["vegetarian"] and r["minutes"] <= 30 for r in page), "the text did not act as a filter"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

rules in the text:
  0.557  Herby beef green curry            Thai                35 min
  0.556  Herby turkey green curry          Thai                30 min
  0.555  Lemon garlic beef green curry     Thai                45 min
  0.547  Lemon garlic turkey green curry   Thai                30 min
  0.542  Herby lamb green curry            Thai                40 min
AssertionError: the text did not act as a filter


All five are meat curries and three take longer than 30 minutes, so it prints `AssertionError:
the text did not act as a filter`. Fix: keep the taste in the text and pass the rules as
arguments:

In [26]:
show(find("a hot curry", vegetarian=True, max_minutes=30), "rules as arguments:")

rules as arguments:
  0.560  Spicy tofu green curry            Thai           veg  25 min
  0.536  Smoky tofu green curry            Thai           veg  30 min
  0.503  Herby tempeh green curry          Thai           veg  30 min
  0.502  Creamy tempeh curry               Indian         veg  30 min
  0.482  Spicy spinach green curry         Thai           veg  25 min


**3. Calling `query_points` with a lowercase keyword.** Keyword matches are exact, so a filter
built by hand around R5's check returns an empty page, with no error to warn you:

In [27]:
try:
    thai = models.Filter(must=[models.FieldCondition(key="cuisine", match=models.MatchValue(value="thai"))])
    hits = client.query_points("recipes", query=embed_query("a hot curry"), query_filter=thai).points
    assert hits, "cuisine='thai' matched nothing"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AssertionError: cuisine='thai' matched nothing


It prints `AssertionError: cuisine='thai' matched nothing`. Fix: go through `find`, whose
`check_args` rejects unknown values, or normalise the input (`"thai".title()` gives `"Thai"`)
before building the filter.

In [28]:
print([r["title"] for r in find("a hot curry", k=3, cuisine="thai".title())])

['Spicy tofu green curry', 'Spicy pork green curry', 'Spicy tempeh green curry']


## Try it
1. **Several cuisines.** Let `cuisine` also accept a list, e.g. `cuisine=["Indian", "Thai"]`, using
   `MatchAny` (lesson 11), and validate every name. Print `find("a hot curry", cuisine=["Indian",
   "Thai"], vegetarian=True)` and check every hit is Indian or Thai.
2. **More like this, with the same rules.** Write `find_like(slug, k=5, **rules)` that queries by
   the point id `recipe_id(slug)` (lesson 10) with the same filters as `find`. Try
   `find_like("beef-stew", vegetarian=True, max_minutes=30)`: which quick vegetarian recipes are
   closest to Beef stew?

In [29]:
# your code here

Clean up: close the local client (the `:memory:` collection disappears with it).

In [30]:
client.close()
print("client closed")

client closed


## Key terms
- **hard constraint** — a rule every result must satisfy (vegetarian, ≤ 30 minutes); it belongs in a payload filter.
- **soft preference** — a wish about meaning or taste ("spicy", "comforting"); it belongs in the query text and is ranked by the embedding.
- **filter builder** — a function that turns optional keyword arguments into a `models.Filter`, adding one condition per rule given and none for `None`.
- **negation blindness** — embedding models largely ignore words like "not"; put exclusions in `must_not` rules.
- **filter test** — a test that checks every hit obeys its filters over many queries and rule combinations; it proves hits are allowed, not good.